In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_excel("/Users/mac/Desktop/enve/final enve/final_filled_climate.xlsx")

# Convert date
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%d/%m/%Y"
)

# =====================================================
# 1. CHECK PRESSURE ZEROS
# =====================================================

print("Pressure zeros:",
      (df["PRESS"] == 0).sum())

# Replace pressure zeros with NaN
df.loc[df["PRESS"] == 0, "PRESS"] = np.nan

# Interpolate pressure station-wise
df["PRESS"] = df.groupby("Station")["PRESS"].transform(
    lambda x: x.interpolate(method="linear")
)

# Fill remaining edge NaNs
df["PRESS"] = df.groupby("Station")["PRESS"].transform(
    lambda x: x.bfill().ffill()
)

print("Remaining PRESS NaN:",
      df["PRESS"].isna().sum())

# =====================================================
# 2. CHECK NABLUS SUNSHINE
# =====================================================

nablus_station = "NAB00003"

print("\nNablus Sunshine Statistics")
print(
    df[df["Station"] == nablus_station]["SUNSHINE"].describe()
)

# Count sunshine zeros
print(
    "Nablus sunshine zeros:",
    (
        (df["Station"] == nablus_station)
        &
        (df["SUNSHINE"] == 0)
    ).sum()
)

# =====================================================
# 3. CREATE MONTH COLUMN
# =====================================================

df["Month"] = df["Date"].dt.month

# =====================================================
# 4. MONTHLY CLIMATOLOGY FROM OTHER STATIONS
# =====================================================

monthly_sunshine = (
    df[df["Station"] != nablus_station]
    .groupby("Month")["SUNSHINE"]
    .mean()
)

print("\nMonthly Sunshine Climatology")
print(monthly_sunshine)

# =====================================================
# 5. REPLACE NABLUS SUNSHINE ZEROS
# =====================================================

mask = (
    (df["Station"] == nablus_station)
    &
    (df["SUNSHINE"] == 0)
)

for month in range(1, 13):

    monthly_value = monthly_sunshine.loc[month]

    df.loc[
        mask & (df["Month"] == month),
        "SUNSHINE"
    ] = monthly_value

# =====================================================
# 6. FINAL CHECKS
# =====================================================

print("\nAfter Cleaning")

print(
    "Pressure zeros:",
    (df["PRESS"] == 0).sum()
)

print(
    "Nablus sunshine zeros:",
    (
        (df["Station"] == nablus_station)
        &
        (df["SUNSHINE"] == 0)
    ).sum()
)

print(
    "Missing values:\n",
    df.isnull().sum()
)

# =====================================================
# 7. SAVE CLEAN DATASET
# =====================================================

df.to_excel(
    "final_filled_climate_cleaned.xlsx",
    index=False
)

print(
    "\nSaved as final_filled_climate_cleaned.xlsx"
)